# 01 · A first look at ERA5 over India (Phase 1, task 1.1.2)

Exploration only, never imported by the pipeline. Reads the one-day test download
(15 Jan 2026, padded study box) made with
`uv run stratum download-era5 --month 2026-01 --days 15` and checks the ranges are sane:
temperature at 250 hPa ≈ 215–240 K over India in January; RHi mostly < 1 with patches above.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from stratum.config import corridor_polygon, load_domain
from stratum.paths import drive_root, repo_root
from stratum.physics.thermo import rhi_from_q

ds = xr.open_zarr(drive_root() / "raw/era5/pl/2026-01_days-15-15/data.zarr")
print(ds)

In [ ]:
s = ds.sel(level=250.0, time="2026-01-15T06:00")  # 11:30 IST
T = s["air_temperature"]
rhi = rhi_from_q(s["specific_humidity"].values, T.values, 25000.0)
print(f"T: {float(T.min()):.1f}–{float(T.max()):.1f} K | RHi median {np.median(rhi):.2f}, "
      f"share ≥ 1: {(rhi >= 1).mean():.1%}")

In [ ]:
# Sanity plot: T and RHi at 250 hPa, with the study box and the Delhi–Mumbai corridor.
dom = load_domain()
corr = corridor_polygon(dom.corridor)
lon, lat = ds.longitude.values, ds.latitude.values
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
m0 = axes[0].pcolormesh(lon, lat, T.values.T, cmap="cividis", shading="auto")
fig.colorbar(m0, ax=axes[0], label="T (K)")
m1 = axes[1].pcolormesh(lon, lat, rhi.T, cmap="cividis", vmin=0, vmax=1.2, shading="auto")
axes[1].contour(lon, lat, rhi.T, levels=[1.0], colors="white", linewidths=1.2)
fig.colorbar(m1, ax=axes[1], label="RHi (white line: RHi = 1, ice saturation)")
for ax, title in zip(axes, ["Temperature, 250 hPa", "RH over ice, 250 hPa (Murphy & Koop)"]):
    b = dom.study_bbox
    ax.plot([b.lon[0], b.lon[1], b.lon[1], b.lon[0], b.lon[0]],
            [b.lat[0], b.lat[0], b.lat[1], b.lat[1], b.lat[0]], "k--", lw=0.8)
    ax.plot(*corr.exterior.xy, color="#c0392b", lw=1.2)
    ax.set_title(f"{title} — ERA5, 2026-01-15 06 UTC", fontsize=10)
    ax.set_xlabel("lon (°E)"); ax.set_ylabel("lat (°N)")
out = repo_root() / "reports/phase1/era5_250hPa_2026-01-15T06.png"
fig.savefig(out, dpi=130)
print("saved", out)